# Rotate API keys → push to the VM

**One-shot notebook for rotating Bybit API keys on the trading VM.**

What this does:
1. Reads your API keys + connection details from Colab Secrets.
2. Reads your SSH private key from a file you upload to the Colab Files panel.
3. Generates a fresh `.env.live` matching the bot's lookup contract.
4. Uploads it to the VM via SSH (atomic write, owner-only permissions).
5. Restarts the trader systemd unit and verifies it's active.

**How to run:**
1. Drag your VM SSH private key file into the Colab **Files** panel (left sidebar). Default filename: `vm_ssh_key` (the notebook also accepts `id_rsa`, `id_ed25519`, `id_ecdsa`).
2. `Runtime → Run all`.

**Required Colab Secrets** (set them once via `Tools → Secrets`, toggle *Notebook access* on):

| Name | What it holds |
|---|---|
| `BYBIT_API_KEY_1`     | Bybit API key for account `bybit_1` (turtle_soup) |
| `BYBIT_API_SECRET_1`  | Bybit API secret for account `bybit_1` |
| `BYBIT_API_KEY_2`     | Bybit API key for account `bybit_2` (vwap) |
| `BYBIT_API_SECRET_2`  | Bybit API secret for account `bybit_2` |
| `TELEGRAM_BOT_TOKEN`  | Bot token from @BotFather |
| `TELEGRAM_CHAT_ID`    | Your Telegram chat id |
| `VM_SSH_HOST`         | The VM's hostname or public IP |
| `VM_SSH_USER`         | SSH user on the VM (usually `ubuntu`) |

**Required file** (drag-drop into the Colab Files sidebar):

| Filename | What it is |
|---|---|
| `vm_ssh_key` (or `id_rsa` / `id_ed25519`) | Your VM SSH **private** key file. Drag it from your laptop into the Files panel. Don't open or rename it — Colab keeps the contents private to your session. |

**Optional Colab Secrets** (skip if unused):

| Name | What it holds |
|---|---|
| `BREAKOUT_API_KEY_1`    | Prop-firm API key (currently disabled in `accounts.yaml`) |
| `BREAKOUT_API_SECRET_1` | Prop-firm API secret |
| `NEWS_API_KEY`          | NewsAPI key (only if you've enabled the news layer) |

---

**Security:** Colab Secrets are scoped to your Google account and never appear in the notebook source. The uploaded SSH key file lives only in this Colab session's `/content/` directory and is wiped when the runtime ends. This notebook never prints, logs, or commits any secret value.

**When to run:** when you rotate keys, change the SSH key, or first set up the VM. The bot keeps using the existing `.env.live` until you re-run.

In [ ]:
# Step 1: Load Colab Secrets + locate the SSH key file.
import os
from google.colab import userdata

REQUIRED = [
    'BYBIT_API_KEY_1', 'BYBIT_API_SECRET_1',
    'BYBIT_API_KEY_2', 'BYBIT_API_SECRET_2',
    'TELEGRAM_BOT_TOKEN', 'TELEGRAM_CHAT_ID',
    'VM_SSH_HOST', 'VM_SSH_USER',
]
OPTIONAL = [
    'BREAKOUT_API_KEY_1', 'BREAKOUT_API_SECRET_1',
    'NEWS_API_KEY',
]

# SSH key candidates — first match wins. Drop your private key into
# /content/ via the Colab Files sidebar with one of these names.
SSH_KEY_CANDIDATES = [
    '/content/vm_ssh_key',
    '/content/id_rsa',
    '/content/id_ed25519',
    '/content/id_ecdsa',
]

secrets = {}
missing = []
for name in REQUIRED:
    try:
        v = userdata.get(name)
    except Exception:
        v = None
    if not v:
        missing.append(name)
    else:
        secrets[name] = v

for name in OPTIONAL:
    try:
        v = userdata.get(name)
        if v:
            secrets[name] = v
    except Exception:
        pass

if missing:
    raise SystemExit(
        'Missing required Colab Secrets:\n  - '
        + '\n  - '.join(missing)
        + '\n\nFix: Tools → Secrets, add each missing secret with exactly that name '
        + '(toggle Notebook access on), then run all again.'
    )

ssh_key_path = next((p for p in SSH_KEY_CANDIDATES if os.path.exists(p)), None)
if ssh_key_path is None:
    raise SystemExit(
        'SSH key file not found. Drag your VM SSH private key into the '
        'Colab Files panel (left sidebar) and name it one of:\n  - '
        + '\n  - '.join(SSH_KEY_CANDIDATES)
        + '\n\nDefault: vm_ssh_key (no extension). Then run all again.'
    )

# Quick sanity check that the file looks like a private key, not a
# public key or random text.
with open(ssh_key_path, 'r') as f:
    first_line = f.readline().strip()
if not first_line.startswith('-----BEGIN'):
    raise SystemExit(
        f'{ssh_key_path} does not look like a private key (first line: '
        f'\"{first_line[:40]}\"). Expected something like \"-----BEGIN '
        f'OPENSSH PRIVATE KEY-----\". Did you accidentally upload your '
        f'public key (.pub) instead of the private one?'
    )
if 'PUBLIC KEY' in first_line:
    raise SystemExit(
        f'{ssh_key_path} is a PUBLIC key. Upload the matching PRIVATE key '
        f'(no .pub extension).'
    )

loaded_required = [n for n in REQUIRED if n in secrets]
loaded_optional = [n for n in OPTIONAL if n in secrets]
print(f'Loaded {len(loaded_required)} required + {len(loaded_optional)} optional secrets from Colab.')
print(f'  Required:  {sorted(loaded_required)}')
print(f'  Optional:  {sorted(loaded_optional) or "(none)"}')
print(f'  SSH key :  {ssh_key_path} (looks like a valid private key)')
print()
print(f'VM target: {secrets["VM_SSH_USER"]}@{secrets["VM_SSH_HOST"]}')

In [ ]:
# Step 2: Build the .env.live content.
#
# Matches the keys produced by scripts/render_env_from_master.py for the
# vwap_btcusd_live profile. Risk caps are hardcoded here to the production
# defaults from config/master-secrets.template.yaml::risk.vwap_btcusd. Edit
# this cell if you intentionally want to change a non-secret default.

PRODUCTION_DEFAULTS = {
    'ENVIRONMENT': 'production',
    'EXCHANGE': 'bybit',
    'MODE': 'LIVE',
    'DRY_RUN': 'false',
    'ALLOW_LIVE_TRADING': 'true',
    'BYBIT_TESTNET': 'false',
    'STRATEGY': 'vwap',
    'SYMBOL': 'BTCUSDT',
    'TIMEFRAME': '1m',
    'DATA_DIR': 'data/',
    'MODEL_DIR': 'ml/models/',
    'LOG_DIR': 'logs/',
    'DB_PATH': 'data/trading.db',
    'MAX_POSITION_USD': '50',
    'MAX_DAILY_LOSS_USD': '25',
    'RISK_PER_TRADE': '0.005',
    'MAX_QTY': '0.001',
    'MAX_OPEN_POSITIONS': '1',
    'NEWS_ENABLED': 'false',
    'NEWS_API_KEY': '',
    # Legacy single-account vars (back-compat with code paths that still
    # read the unsuffixed names; mirrors render_env_from_master.py).
    'BYBIT_API_KEY': secrets['BYBIT_API_KEY_1'],
    'BYBIT_API_SECRET': secrets['BYBIT_API_SECRET_1'],
    'BYBIT_BASE_URL': 'https://api.bybit.com',
}

# Per-account credentials (the names accounts.yaml looks up via api_key_env).
PER_ACCOUNT_KEYS = [
    'BYBIT_API_KEY_1', 'BYBIT_API_SECRET_1',
    'BYBIT_API_KEY_2', 'BYBIT_API_SECRET_2',
    'BREAKOUT_API_KEY_1', 'BREAKOUT_API_SECRET_1',
]

TELEGRAM_KEYS = ['TELEGRAM_BOT_TOKEN', 'TELEGRAM_CHAT_ID']


def quote_if_needed(val: str) -> str:
    if any(c in val for c in ' \t#$`\'"\\'):
        return '"' + val.replace('\\', '\\\\').replace('"', '\\"') + '"'
    return val


lines = []
for k, v in PRODUCTION_DEFAULTS.items():
    if k == 'NEWS_API_KEY' and 'NEWS_API_KEY' in secrets:
        v = secrets['NEWS_API_KEY']
    if k == 'NEWS_ENABLED' and 'NEWS_API_KEY' in secrets:
        v = 'true'
    lines.append(f'{k}={quote_if_needed(str(v))}')
for k in TELEGRAM_KEYS:
    lines.append(f'{k}={quote_if_needed(secrets[k])}')
for k in PER_ACCOUNT_KEYS:
    if k in secrets:
        lines.append(f'{k}={quote_if_needed(secrets[k])}')

env_content = '\n'.join(lines) + '\n'

# Print only the KEY NAMES, never values. Operator can confirm shape
# without leaking secrets.
key_names = [line.split('=', 1)[0] for line in lines]
print(f'Generated .env.live with {len(key_names)} variables:')
for name in key_names:
    print(f'  {name}')

In [ ]:
# Step 3: Push .env.live to the VM via SCP, restart the trader, verify.
import os
import shutil
import stat
import subprocess
import tempfile

REPO_PATH_ON_VM = '/home/' + secrets['VM_SSH_USER'] + '/ict-trading-bot'
REMOTE_FINAL = REPO_PATH_ON_VM + '/.env.live'
REMOTE_TMP = REMOTE_FINAL + '.tmp'
SERVICE_NAME = 'ict-trader-live.service'

# Stage SSH key + .env.live in a Colab tempdir with 0600 perms — ssh
# refuses to use a key file with broader permissions.
tmpdir = tempfile.mkdtemp(prefix='rotate-keys-')
key_path = os.path.join(tmpdir, 'vm_key')
env_path = os.path.join(tmpdir, '.env.live')

shutil.copy(ssh_key_path, key_path)
os.chmod(key_path, stat.S_IRUSR | stat.S_IWUSR)  # 0600

with open(env_path, 'w') as f:
    f.write(env_content)
os.chmod(env_path, stat.S_IRUSR | stat.S_IWUSR)

host = secrets['VM_SSH_HOST']
user = secrets['VM_SSH_USER']
ssh_target = f'{user}@{host}'
ssh_opts = [
    '-i', key_path,
    '-o', 'StrictHostKeyChecking=accept-new',
    '-o', 'UserKnownHostsFile=' + os.path.join(tmpdir, 'known_hosts'),
    '-o', 'ConnectTimeout=15',
    '-o', 'BatchMode=yes',
]


def _redact(text: str) -> str:
    """Strip any secret values that may have leaked into stderr."""
    if not text:
        return text
    for v in secrets.values():
        if v and len(str(v)) > 8:
            text = text.replace(str(v), '<REDACTED>')
    return text


def run_ssh(cmd: str, *, label: str):
    full = ['ssh'] + ssh_opts + [ssh_target, cmd]
    proc = subprocess.run(full, capture_output=True, text=True)
    if proc.returncode != 0:
        print(f'❌ {label} failed (exit {proc.returncode})')
        print(_redact(proc.stderr or proc.stdout)[:500])
        raise SystemExit(f'{label} failed')
    return (proc.stdout or '').strip()


def run_scp(local: str, remote: str, *, label: str):
    full = ['scp'] + ssh_opts + [local, f'{ssh_target}:{remote}']
    proc = subprocess.run(full, capture_output=True, text=True)
    if proc.returncode != 0:
        print(f'❌ {label} failed (exit {proc.returncode})')
        print(_redact(proc.stderr or proc.stdout)[:500])
        raise SystemExit(f'{label} failed')


try:
    print(f'\u23f3 Connecting to {ssh_target} …')
    run_ssh('echo connected', label='SSH connectivity')
    print('✅ SSH OK')

    print(f'\u23f3 Uploading new .env.live (atomic) …')
    run_scp(env_path, REMOTE_TMP, label='SCP upload')
    run_ssh(f'chmod 600 {REMOTE_TMP} && mv {REMOTE_TMP} {REMOTE_FINAL}',
            label='atomic rename')
    print(f'✅ Wrote {REMOTE_FINAL}')

    print(f'\u23f3 Restarting {SERVICE_NAME} …')
    run_ssh(f'sudo -n systemctl restart {SERVICE_NAME}',
            label='service restart')
    print(f'✅ Restart issued')

    print(f'\u23f3 Verifying service is active …')
    state = run_ssh(f'sudo -n systemctl is-active {SERVICE_NAME}',
                    label='is-active check')
    print(f'✅ Service state: {state}')
    if state != 'active':
        print('⚠️  Service is not "active" — check journalctl on the VM.')
finally:
    # Wipe the SSH key copy + env file from the Colab tmpdir.
    # The original /content/vm_ssh_key is left in place so you can
    # re-run the notebook without re-uploading. Delete it manually
    # from the Files panel when you're done if you want.
    shutil.rmtree(tmpdir, ignore_errors=True)

print('\nDone. Open Telegram and run /accounts_status to verify each account shows a real balance.')

## Verification

After the cells above complete, in Telegram:

1. **`/accounts_status`** — every account should show ✅ with a real USDT balance.
2. **`/smoke_test`** — each account should return ✅ `rejected_too_small` (Bybit accepted the request and rejected for size, which proves the keys work).

If any account still shows ❌, the new diagnostic message names the exact reason (missing env var, `Bybit error retCode=10003`, network, etc.). See the troubleshooting section in [`docs/operator/colab-key-rotation.md`](https://github.com/the-lizardking/ict-trading-bot/blob/main/docs/operator/colab-key-rotation.md).

## When to re-run

- You rotated a Bybit API key.
- You changed the VM's SSH key.
- You added a new account to `config/accounts.yaml`.
- The bot is alerting on `bybit_get_wallet_balance_failed` with `retCode=10003` (key invalid).

## Cleaning up

The uploaded SSH key file in `/content/` is wiped automatically when this Colab session ends (close the tab or `Runtime → Disconnect and delete runtime`). No further cleanup needed.